[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/facufan07/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)

# TPI — Análisis de Churn en Telecomunicaciones

**Materia:** Análisis de Datos  
**Integrantes:** Nicolás Pérez · Lola Gonzalez Suarez · Lucas Gerez · Facundo Fandiño  
**Grupo:** 3  
**División:** 343  
**Dataset:** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

## 0. Configuración del entorno

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Fundamentos

### 1.1 Problema de negocio
La empresa está afrontando una pérdida de ingresos cada vez que sus clientes cancelan sus suscripciones. Esto la afecta porque, además de dejar de cobrar lo que pagaban esos clientes, debe conseguir otros nuevos para compensar las bajas. Por eso, en este trabajo vamos a averiguar, a través de los datos, qué patrones tienen en común los clientes que cancelan, para que la empresa pueda identificar a quienes tienen mayor riesgo de irse y tomar medidas para retenerlos.

### 1.2 Hipótesis analíticas
H1 — Tipo de contrato (Contract): los clientes con contrato mes a mes cancelan más que los clientes con contrato de uno o dos años, porque tienen un compromiso menor y pueden irse con más facilidad.

H2 — Antigüedad (tenure): cuanto mayor es la antigüedad del cliente, menos probable es que cancele, porque quien lleva más tiempo en la empresa ya está más acostumbrado al servicio y tiene menos motivos para buscar otro.

H3 — Cargos mensuales (MonthlyCharges): los clientes con cargos mensuales más altos cancelan más que los clientes con cargos más bajos, porque el costo del servicio pesa más en su presupuesto y tienen más incentivos para buscar una alternativa.

H4 — Personas a cargo (Dependents): los clientes sin personas a cargo cancelan más que los clientes con personas a cargo, porque el servicio suele afectar a menos personas y les resulta más simple cambiar de proveedor.

### 1.3 Variable objetivo
La variable objetivo es Churn, que indica si un cliente canceló su suscripción al servicio. Es una variable binaria: toma el valor Yes cuando el cliente canceló y No cuando no lo hizo. Es la variable central del análisis porque representa el problema que enfrenta la empresa: al compararla con las demás variables, podemos identificar qué características tienen en común los clientes que cancelan.

## 2. Carga de datos

In [9]:
# Se lee el CSV desde GitHub para que el notebook corra en Colab sin subir archivos.
DATA_URL = "https://raw.githubusercontent.com/facufan07/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_raw = pd.read_csv(DATA_URL)
df_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 2.1 Clasificación del dataset y de las variables
_Tipo de dato según su estructura (estructurado / semi-estructurado / no estructurado) y relevamiento de variables (nominales, ordinales, discretas, continuas), contrastado con los `dtypes` de Pandas._

### La estructura del conjunto de datos es estructurada.

| Variable | Tipo de dato según su estructura | Dtype en Pandas |
|---|---|---|
| **customerID** | Cualitativa nominal | `object` |
| **gender** | Cualitativa nominal | `object` |
| **SeniorCitizen** | Cualitativa nominal  | `int64` |
| **Partner** | Cualitativa nominal | `object` |
| **Dependents** | Cualitativa nominal | `object` |
| **tenure** | Cuantitativa discreta | `int64` |
| **PhoneService** | Cualitativa nominal | `object` |
| **MultipleLines** | Cualitativa nominal | `object` |
| **InternetService** | Cualitativa nominal | `object` |
| **OnlineSecurity** | Cualitativa nominal | `object` |
| **OnlineBackup** | Cualitativa nominal | `object` |
| **DeviceProtection** | Cualitativa nominal | `object` |
| **TechSupport** | Cualitativa nominal | `object` |
| **StreamingTV** | Cualitativa nominal | `object` |
| **StreamingMovies** | Cualitativa nominal | `object` |
| **Contract** | Cualitativa ordinal | `object` |
| **PaperlessBilling** | Cualitativa nominal | `object` |
| **PaymentMethod** | Cualitativa nominal | `object` |
| **MonthlyCharges** | Cuantitativa continua | `float64` |
| **TotalCharges** | Cuantitativa continua | `object` |
| **Churn** | Cualitativa nominal | `object` |

In [10]:
df_raw.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


## 3. Manipulación de datos

### 3.1 Selección y filtrado con múltiples condiciones (`&`, `|`)

In [26]:
# Consulta Filtrada n° 1 (AND): contrato mensual y antigüedad de hasta 12 meses
f1 = df_raw[(df_raw["Contract"] == "Month-to-month") & (df_raw["tenure"] <= 12)]
print(f"Personas con contrato mensual y antigüedad de hasta 12 meses: {len(f1)} clientes ({len(f1) / len(df_raw):.1%} del total)")

# Consulta Filtrada n° 2 (AND): cargo mensual alto y cliente que canceló
f2 = df_raw[(df_raw["MonthlyCharges"] > 70) & (df_raw["Churn"] == "Yes")]
print(f"Personas con cargo mensual alto y cliente que canceló: {len(f2)} clientes ({len(f2) / len(df_raw):.1%} del total)")

# Consulta Filtrada n°3 (OR): fibra óptica o pago con cheque electrónico
f3 = df_raw[(df_raw["InternetService"] == "Fiber optic") | (df_raw["InternetService"] == "DSL")]
print(f"Personas con fibra óptica o pago con cheque electrónico: {len(f3)} clientes ({len(f3) / len(df_raw):.1%} del total)")

# Consulta Filtrada n°4 (AND): clientes sin personas a cargo que cancelaron
f4 = df_raw[(df_raw["Dependents"] == "No") & (df_raw["Churn"] == "Yes")]
print(f"Clientes sin personas a cargo que cancelaron: {len(f4)} clientes ({len(f4) / len(df_raw):.1%} del total)")

# Consulta Filtrada n°5 (AND & OR)
f5 = df_raw[(df_raw["Churn"] == "Yes") & ((df_raw["OnlineSecurity"] == "No") | (df_raw["TechSupport"] == "No"))]
print(f"Clientes que cancelaron y no tienen internet o soporte técnico: {len(f5)} clientes ({len(f5) / len(df_raw):.1%} del total)")



Personas con contrato mensual y antigüedad de hasta 12 meses: 1994 clientes (28.3% del total)
Personas con cargo mensual alto y cliente que canceló: 1267 clientes (18.0% del total)
Personas con fibra óptica o pago con cheque electrónico: 5517 clientes (78.3% del total)
Clientes sin personas a cargo que cancelaron: 1543 clientes (21.9% del total)
Clientes que cancelaron y no tienen internet o soporte técnico: 1657 clientes (23.5% del total)


### 3.2 Agregación con `groupby` (múltiples agregaciones)

Filtro con &: 991 de 7043 clientes


### 3.3 Transformación básica
_Renombrado de columnas, eliminación de atributos irrelevantes y creación de columnas calculadas._

## 4. Limpieza y calidad de datos

### 4.1 Valores faltantes y cadenas anómalas
_Incluye los espacios vacíos en `TotalCharges`._

### 4.2 Adecuación de tipos de datos (`dtypes`)

### 4.3 Valores atípicos (boxplot / IQR)

## 5. Análisis exploratorio

### 5.1 Análisis cuantitativo
_Tendencia central, dispersión y correlación (Pearson / Spearman) entre `tenure`, `MonthlyCharges` y `TotalCharges`._

### 5.2 Análisis categórico
_Frecuencias absolutas y relativas, y tablas de contingencia respecto a `Churn`._

## 6. Visualización
_Cada gráfico va acompañado de una interpretación orientada al negocio._

### 6.1 Distribución de `tenure` y `MonthlyCharges` (histograma / KDE)

### 6.2 Matriz de correlación (heatmap)

### 6.3 Tasa de churn por tipo de contrato y método de pago

## 7. Preprocesamiento para modelado
_Codificación de categóricas (One-Hot / Ordinal), escalado de numéricas y partición estratificada 80/20. Los transformadores se ajustan solo sobre el conjunto de entrenamiento para evitar data leakage._

## 8. Regresión lineal simple
_Estimación de una variable continua (por ejemplo, `TotalCharges`). Métricas: R² y MAE._

## 9. Conclusiones
_Completar: hallazgos principales y recomendaciones para el negocio._